In [1]:
import os 

from  PIL import Image
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms, models
from sklearn.metrics import accuracy_score

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
class PneumoniaDataset(Dataset):
    def __init__(self, data_dir, transform=None):
        self.data_dir = data_dir
        self.transform = transform
        self.image_paths = []
        self.labels = []
        
        for label in ["NORMAL", "PNEUMONIA"]:
            class_dir = os.path.join(data_dir, label)
            for image_name in os.listdir(class_dir):
                self.image_paths.append(os.path.join(class_dir, image_name))
                self.labels.append(0 if label == "NORMAL" else 1)
                
        
    def __len__(self):
        return len(self.image_paths)
    
    
    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        img = Image.open(img_path).convert('RGB')
        label = self.labels[idx]
        
        if self.transform:
            img = self.transform(img)
            
        return img, label

In [3]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
]) 


train_dataset = PneumoniaDataset(data_dir='chest_xray/train', transform=transform)
test_dataset = PneumoniaDataset(data_dir='chest_xray/test', transform=transform)
val_dataset = PneumoniaDataset(data_dir='chest_xray/val', transform=transform)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

In [4]:
model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
model.fc = nn.Linear(model.fc.in_features, 2)
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

epochs = 10

for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    
    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
        
    print(f'Epoch {epoch+1}, loss: {running_loss/len(train_loader)}')
    
    model.eval()  
    val_labels = []
    val_preds = []
    
    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(device)
            labels = labels.to(device)
            
            outputs = model(images)
            _, preds = torch.max(outputs, 1)
            
            val_labels.extend(labels.cpu().numpy())
            val_preds.extend(preds.cpu().numpy())
            
    val_acc = accuracy_score(val_labels, val_preds)
    
    print(f'Validation accuracy: {val_acc}')
    
model.eval()

test_labels = []
test_preds = []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)
        _, preds = torch.max(outputs, 1)

        test_labels.extend(labels.cpu().numpy())
        test_preds.extend(preds.cpu().numpy())

test_acc = accuracy_score(test_labels, test_preds)
print("Test accuracy:" ,test_acc)

torch.save(model.state_dict(), 'pneumonia_model.pth')

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /Users/umayyentur/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth
100%|██████████| 44.7M/44.7M [00:01<00:00, 28.4MB/s]


Epoch 1, loss: 0.11558014880637267
Validation accuracy: 0.9375
Epoch 2, loss: 0.0668979803738727
Validation accuracy: 0.625
Epoch 3, loss: 0.03354454460158156
Validation accuracy: 0.8125
Epoch 4, loss: 0.04824501460901443
Validation accuracy: 0.625
Epoch 5, loss: 0.03284164793229451
Validation accuracy: 1.0
Epoch 6, loss: 0.02616465598138755
Validation accuracy: 0.625
Epoch 7, loss: 0.031334886682881154
Validation accuracy: 1.0
Epoch 8, loss: 0.022117280453649576
Validation accuracy: 0.9375
Epoch 9, loss: 0.03192582568273368
Validation accuracy: 0.9375
Epoch 10, loss: 0.0362118412403067
Validation accuracy: 1.0
Test accuracy: 0.7852564102564102
